## Linear Regression Model

In [2]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score, accuracy_score, f1_score

In [3]:
df = pd.read_csv("data_train.csv")

In [4]:
selected_features = pd.read_csv('selected_features.csv')
selected_features = selected_features.iloc[:, 0].tolist()
len(selected_features)

35

In [5]:
# Full feature set (drop IDs + targets)
full_feature_cols = [
    c for c in df.columns
    if c not in ["stock_id", "date", "R1M_Usd", "R1M_Usd_C"]
]
len(full_feature_cols)

93

In [6]:
# Time-based split *within* 2000–2007 for NN tuning
df["date"] = pd.to_datetime(df["date"])

cutoff_date = pd.Timestamp("2006-01-01")

df_train = df[df["date"] <  cutoff_date].copy()   # 2000–2005
df_val   = df[df["date"] >= cutoff_date].copy()   # 2006–2007

print(df_train["date"].min(), df_train["date"].max())
print(df_val["date"].min(),   df_val["date"].max())

1999-12-31 00:00:00 2005-12-31 00:00:00
2006-01-31 00:00:00 2007-12-31 00:00:00


In [7]:
# Full and selected feature sets

X_train_full = df_train[full_feature_cols].copy()
X_val_full   = df_val[full_feature_cols].copy()

X_train_sel  = df_train[selected_features].copy()
X_val_sel    = df_val[selected_features].copy()

y_train_reg = df_train["R1M_Usd"].copy()
y_val_reg   = df_val["R1M_Usd"].copy()

In [8]:
# Fitting Linear Regression on FULL features

lr_full = LinearRegression()
lr_full.fit(X_train_full, y_train_reg)

y_pred_train_full = lr_full.predict(X_train_full)
y_pred_val_full   = lr_full.predict(X_val_full)

## mse and r2 for FULL features
mse_train_full = mean_squared_error(y_train_reg, y_pred_train_full)
mse_val_full  = mean_squared_error(y_val_reg,  y_pred_val_full)
r2_train_full  = r2_score(y_train_reg, y_pred_train_full)
r2_val_full   = r2_score(y_val_reg,  y_pred_val_full)

In [9]:
# Fitting Linear Regression on SELECTED features

lr_sel = LinearRegression()
lr_sel.fit(X_train_sel, y_train_reg)

y_pred_train_sel = lr_sel.predict(X_train_sel)
y_pred_val_sel   = lr_sel.predict(X_val_sel)

mse_train_sel = mean_squared_error(y_train_reg, y_pred_train_sel)
mse_val_sel   = mean_squared_error(y_val_reg,  y_pred_val_sel)
r2_train_sel  = r2_score(y_train_reg, y_pred_train_sel)
r2_val_sel    = r2_score(y_val_reg,  y_pred_val_sel)

In [10]:
## Measuring classification performance by sign of R1M_Usd and assigning 1 for positive returns and 0 for negative returns

y_train_cls = df_train["R1M_Usd_C"]
y_val_cls  = df_val["R1M_Usd_C"]

## Assigning 0, 1
y_pred_train_full_cls = (y_pred_train_full > 0).astype(int)
y_pred_val_full_cls  = (y_pred_val_full  > 0).astype(int)

y_pred_train_sel_cls = (y_pred_train_sel > 0).astype(int)
y_pred_val_sel_cls  = (y_pred_val_sel  > 0).astype(int)

## Checking on AUC
acc_train_full = accuracy_score(y_train_cls, y_pred_train_full_cls)
acc_val_full  = accuracy_score(y_val_cls,  y_pred_val_full_cls)

acc_train_sel = accuracy_score(y_train_cls, y_pred_train_sel_cls)
acc_val_sel  = accuracy_score(y_val_cls,  y_pred_val_sel_cls)

In [11]:
metrics_df = pd.DataFrame({
    "model": ["LinearReg_full", "LinearReg_selected"],
    "mse_train": [mse_train_full, mse_train_sel],
    "mse_val":  [mse_val_full,  mse_val_sel],
    "r2_train":  [r2_train_full,  r2_train_sel],
    "r2_val":   [r2_val_full,   r2_val_sel],
    "acc_train": [acc_train_full, acc_train_sel],
    "acc_val":  [acc_val_full,  acc_val_sel],
})

metrics_df

,model,mse_train,mse_val,r2_train,r2_val,acc_train,acc_val
0,LinearReg_full,0.027406,0.016321,0.012771,-0.020959,0.517322,0.495954
1,LinearReg_selected,0.027509,0.016291,0.009058,-0.019062,0.513954,0.493478


## Backtest - Rebalancing Loop

In [12]:
data = pd.read_csv("data_backtest.csv").copy()
data["date"] = pd.to_datetime(data["date"])
data["ym"]   = data["date"].dt.to_period("M")

In [ ]:
backtest_start = "2008-01-01"
backtest_end   = "2018-12-31"

LOOKBACK_MONTHS = [12, 24, 36, 48, 60]   # lookback periods to try

In [14]:
## Function to get month window
def get_month_window(df, current_ym, lookback):
    """
    Use the previous `lookback` months as training window.
    For example, if current_ym = 2008-06 and lookback=12,
    we use 2007-06 ... 2008-05 as training.
    """
    start_ym = current_ym - lookback    # period math
    mask = (df["ym"] > start_ym) & (df["ym"] < current_ym)
    return df[mask].copy()

## Strategy

Long leg = top 20% by model score, equal-weighted.

Short leg = bottom 20% by model score, equal-weighted.

Portfolio return = long_ret + short_ret.

In [ ]:
## ROLLING BACKTEST LOOP (FOR LINEAR REGRESSION)

def run_backtest_linear(df, features, lookback_months):
    monthly_returns = []
    month_index = []

    # unique year-months in backtest range, sorted
    months = (
        df.loc[(df["date"] >= backtest_start) & (df["date"] <= backtest_end), "ym"]
          .sort_values()
          .unique()
    )

    for ym in months:
        # 1) train window: previous `lookback_months` months
        train_df = get_month_window(df, ym, lookback_months)
        # 2) test month = current ym
        test_df  = df[df["ym"] == ym]


        # 3) X, y
        X_train = train_df[features].values
        y_train = train_df["R1M_Usd"].values

        X_test = test_df[features].values
        y_test = test_df["R1M_Usd"].values  # not strictly needed for PnL but fine

        # 4) fit LR
        model = LinearRegression()
        model.fit(X_train, y_train)

        # 5) predict
        preds = model.predict(X_test)

        # 6) build L/S portfolio
        test_df = test_df.copy()
        test_df["pred"] = preds
        test_df = test_df.sort_values("pred")

        n = len(test_df)
        if n < 10:
            continue
        
        ## STRATEGY
         # bottom 20% = short, top 20% = long
        long_df  = test_df.iloc[int(0.8 * n):]      # top 20%
        short_df = test_df.iloc[:int(0.2 * n)]      # bottom 20%

        w_long  = 1.0 / len(long_df)
        w_short = -1.0 / len(short_df)

        long_ret  = (long_df["R1M_Usd"] * w_long).sum()
        short_ret = (short_df["R1M_Usd"] * w_short).sum()

        monthly_returns.append(long_ret + short_ret)
        month_index.append(ym.to_timestamp("M"))     # use month-end as index

    return pd.Series(monthly_returns, index=month_index)


In [16]:
# Run backtest for different lookback periods SLELECTED features

results_storage_selected = {}

for L in LOOKBACK_MONTHS:
    print(f"Running LR Backtest with Lookback {L} months...")
    pnl_series = run_backtest_linear(data, selected_features, L)
    results_storage_selected[f"LR_Lookback_{L}"] = pnl_series

Running LR Backtest with Lookback 12 months...
Running LR Backtest with Lookback 24 months...
Running LR Backtest with Lookback 36 months...
Running LR Backtest with Lookback 48 months...
Running LR Backtest with Lookback 60 months...


In [17]:
# Run backtest for different lookback periods FULL features

results_storage_full = {}

for L in LOOKBACK_MONTHS:
    print(f"Running LR Backtest with Lookback {L} months...")
    pnl_series = run_backtest_linear(data, full_feature_cols, L)
    results_storage_full[f"LR_Lookback_{L}"] = pnl_series

Running LR Backtest with Lookback 12 months...
Running LR Backtest with Lookback 24 months...
Running LR Backtest with Lookback 36 months...
Running LR Backtest with Lookback 48 months...
Running LR Backtest with Lookback 60 months...


In [18]:
results_storage_selected

{'LR_Lookback_12': 2008-01-31   -0.008368
 2008-02-29   -0.008081
 2008-03-31   -0.021827
 2008-04-30    0.027698
 2008-05-31    0.053826
                 ...   
 2018-07-31    0.134632
 2018-08-31    0.000899
 2018-09-30   -0.023788
 2018-10-31   -0.009117
 2018-11-30    0.026123
 Length: 131, dtype: float64,
 'LR_Lookback_24': 2008-01-31   -0.011610
 2008-02-29    0.007616
 2008-03-31    0.014105
 2008-04-30    0.023655
 2008-05-31    0.013735
                 ...   
 2018-07-31    0.118260
 2018-08-31    0.000908
 2018-09-30   -0.018005
 2018-10-31   -0.001766
 2018-11-30    0.014144
 Length: 131, dtype: float64,
 'LR_Lookback_36': 2008-01-31   -0.011814
 2008-02-29    0.007770
 2008-03-31    0.014711
 2008-04-30    0.017546
 2008-05-31   -0.008932
                 ...   
 2018-07-31    0.127394
 2018-08-31    0.000028
 2018-09-30   -0.009789
 2018-10-31   -0.003919
 2018-11-30    0.000718
 Length: 131, dtype: float64,
 'LR_Lookback_48': 2008-01-31   -0.003226
 2008-02-29    0.01726

In [19]:
results_storage_full

{'LR_Lookback_12': 2008-01-31   -0.020621
 2008-02-29   -0.000716
 2008-03-31   -0.012602
 2008-04-30    0.023899
 2008-05-31    0.032648
                 ...   
 2018-07-31   -0.126028
 2018-08-31    0.005642
 2018-09-30   -0.018049
 2018-10-31   -0.018019
 2018-11-30    0.024935
 Length: 131, dtype: float64,
 'LR_Lookback_24': 2008-01-31   -0.026632
 2008-02-29   -0.010761
 2008-03-31    0.026799
 2008-04-30    0.038487
 2008-05-31    0.011883
                 ...   
 2018-07-31    0.126092
 2018-08-31    0.000884
 2018-09-30   -0.018305
 2018-10-31   -0.014623
 2018-11-30    0.004497
 Length: 131, dtype: float64,
 'LR_Lookback_36': 2008-01-31   -0.027249
 2008-02-29   -0.006300
 2008-03-31    0.036257
 2008-04-30    0.038209
 2008-05-31    0.004560
                 ...   
 2018-07-31    0.125859
 2018-08-31    0.002969
 2018-09-30   -0.010347
 2018-10-31   -0.010340
 2018-11-30   -0.009606
 Length: 131, dtype: float64,
 'LR_Lookback_48': 2008-01-31   -0.000646
 2008-02-29    0.00077

In [25]:
## Equally Weighted Portfolio

def build_ew_market_series(df):
    # Restrict to backtest period
    df_bt = df[(df["date"] >= backtest_start) & (df["date"] <= backtest_end)].copy()
    df_bt["ym"] = df_bt["date"].dt.to_period("M")

    # Equal-weight average of R1M_Usd across all stocks each month
    ew = df_bt.groupby("ym")["R1M_Usd"].mean()
    ew.index = ew.index.to_timestamp("M")   # month-end timestamps
    return ew

ew_market = build_ew_market_series(data)

In [26]:
## Performace Metrics

def perf_stats(pnl: pd.Series, ew_market: pd.Series):
    """
    pnl: monthly return series of a strategy (or EW itself)
    ew_market: monthly EW market returns
    """
    # Align with EW
    aligned = pd.concat([pnl.astype(float), ew_market.astype(float)], axis=1, join="inner")
    aligned.columns = ["strategy", "ew"]
    r = aligned["strategy"].dropna()
    if len(r) < 2:
        return None

    # Basic stats
    cum = (1 + r).prod()
    ann_ret = cum**(12 / len(r)) - 1

    ann_vol = r.std() * (12**0.5)
    sharpe = ann_ret / ann_vol if ann_vol > 0 else float("nan")

    # Max drawdown
    cum_curve = (1 + r).cumprod()
    peak = cum_curve.cummax()
    dd = (cum_curve / peak) - 1
    max_dd = dd.min()

    # Hit rate
    hit_rate = (r > 0).mean()

    # Correlation to EW
    corr_ew = aligned["strategy"].corr(aligned["ew"])

    return {
        "n_months": len(r),
        "ann_ret": ann_ret,
        "ann_vol": ann_vol,
        "sharpe": sharpe,
        "max_drawdown": max_dd,
        "hit_rate": hit_rate,
        "corr_to_ew": corr_ew,
    }

In [ ]:
## Comapring the SELECTED feature set results vs Equally Weighted Portfolio

all_strategies = {}

# Linear regressions
for name, series in results_storage_selected.items():
    all_strategies[name] = series

rows = []

# First: EW market itself
ew_stats = perf_stats(ew_market, ew_market)
ew_stats["strategy_name"] = "EW_Market"
rows.append(ew_stats)

# Then: every strategy vs EW
for name, pnl in all_strategies.items():
    stats = perf_stats(pnl, ew_market)
    if stats is None:
        continue
    stats["strategy_name"] = name
    rows.append(stats)

perf_df = pd.DataFrame(rows).set_index("strategy_name")
perf_df = perf_df.sort_values("sharpe", ascending=False)
perf_df


,n_months,ann_ret,ann_vol,sharpe,max_drawdown,hit_rate,corr_to_ew
strategy_name,,,,,,,
LR_Lookback_48,131,0.148980,0.130551,1.141164,-0.115145,0.633588,0.627406
LR_Lookback_60,131,0.161220,0.144536,1.115429,-0.127465,0.625954,0.687592
LR_Lookback_36,131,0.111451,0.118006,0.944446,-0.127784,0.625954,0.483112
LR_Lookback_24,131,0.085733,0.112290,0.763501,-0.129526,0.618321,0.265346
EW_Market,131,0.116382,0.201993,0.576170,-0.484271,0.656489,1.000000
LR_Lookback_12,131,0.067645,0.122639,0.551576,-0.275378,0.541985,0.088572


In [ ]:
## Comapring the FULL feature set results vs Equally Weighted Portfolio

all_strategies = {}

# Linear regressions
for name, series in results_storage_full.items():
    all_strategies[name] = series

rows = []

# EW market itself
ew_stats = perf_stats(ew_market, ew_market)
ew_stats["strategy_name"] = "EW_Market"
rows.append(ew_stats)

# Then: every strategy vs EW
for name, pnl in all_strategies.items():
    stats = perf_stats(pnl, ew_market)
    if stats is None:
        continue
    stats["strategy_name"] = name
    rows.append(stats)

perf_df = pd.DataFrame(rows).set_index("strategy_name")
perf_df = perf_df.sort_values("sharpe", ascending=False)
perf_df


,n_months,ann_ret,ann_vol,sharpe,max_drawdown,hit_rate,corr_to_ew
strategy_name,,,,,,,
LR_Lookback_48,131,0.132646,0.127192,1.042883,-0.132440,0.610687,0.577380
LR_Lookback_60,131,0.123967,0.136625,0.907352,-0.141253,0.587786,0.645145
EW_Market,131,0.116382,0.201993,0.576170,-0.484271,0.656489,1.000000
LR_Lookback_36,131,0.063740,0.123957,0.514211,-0.297175,0.557252,0.468334
LR_Lookback_24,131,0.050999,0.121359,0.420236,-0.183460,0.557252,0.350314
LR_Lookback_12,131,0.019735,0.115009,0.171594,-0.410417,0.488550,0.207102


### Best performace is for LR lookback 48 months